# Creacion del ResourceArn


In [11]:
import shutil, os

os.makedirs("provenance", exist_ok=True)
shutil.copy("/opt/ml/metadata/resource-metadata.json", "provenance/sagemaker-resource-metadata.json")

# Verifica que se copió bien y revisa el ResourceArn
import json
with open("provenance/sagemaker-resource-metadata.json") as f:
    meta = json.load(f)
print(meta.get("ResourceArn"))

arn:aws:sagemaker:us-east-1:492606007513:notebook-instance/analisis-sentimientos


# Instalacion

In [2]:
!pip install datasets mlflow scikit-learn -q

# Importacion del dataset

In [12]:
from datasets import load_dataset

dataset = load_dataset(
    "adilbekovich/Sentiment140Twitter",
    revision="b6037e127257d95b9b23d31f78b264b9ebe697fd"
)

train = dataset["train"]
test = dataset["test"]

print(train.num_rows, test.num_rows)  # debería imprimir 1360000 240000

1360000 240000


# Se convierte a pandas y conserva el indice original

In [13]:
import pandas as pd

train_df = train.to_pandas()
train_df["index"] = train_df.index  # conserva la posición original ANTES de muestrear

print(train_df.shape)
print(train_df.columns.tolist())

(1360000, 3)
['text', 'label', 'index']


# Muestra estratificada de 200.000, semilla 42

In [15]:
from sklearn.model_selection import train_test_split

# Ajusta "label" al nombre real de tu columna de etiqueta
sample_df, _ = train_test_split(
    train_df,
    train_size=200000,
    stratify=train_df["label"],
    random_state=42
)

sample_df = sample_df.reset_index(drop=True)  # el índice original ya quedó guardado en la columna "index"
print(sample_df.shape)
print(sample_df["label"].value_counts())

(200000, 3)
label
1    100019
0     99981
Name: count, dtype: int64


# Construir los 3 folds estratificados

In [16]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

sample_df["fold"] = -1
for fold_num, (_, val_idx) in enumerate(skf.split(sample_df, sample_df["label"])):
    sample_df.loc[val_idx, "fold"] = fold_num

print(sample_df["fold"].value_counts())

fold
1    66667
0    66667
2    66666
Name: count, dtype: int64


# Generar protocol/partitions.csv

In [17]:
partitions = sample_df[["index", "fold"]].sort_values("index").reset_index(drop=True)

os.makedirs("protocol", exist_ok=True)
partitions.to_csv("protocol/partitions.csv", index=False)

print(partitions.shape)  # debe ser (200000, 2)
partitions.head()

(200000, 2)


,index,fold
0,11,0
1,12,1
2,25,2
3,26,1
4,46,1


# Generar protocol/members.csv

In [18]:
members = pd.DataFrame([
    {"member_id": "E01", "notebook_arn": "arn:aws:sagemaker:us-east-1:492606007513:notebook-instance/analisis-sentimientos"},
    # agrega una fila por cada integrante del equipo
])

members = members.sort_values("member_id").reset_index(drop=True)
members.to_csv("protocol/members.csv", index=False)
members

,member_id,notebook_arn
0,E01,arn:aws:sagemaker:us-east-1:492606007513:noteb...


In [9]:
train_df.columns.tolist()

['text', 'label', 'index']

In [10]:
train_df.head()

,text,label,index
0,Closet organizer install complete. Now for th...,0,0
1,Mornin' All!! ....I need to wake up....this w...,1,1
2,@Lega_c ahhhhhhhhhh! he suxxxxxxx! he claimed ...,0,2
3,@endlessblush Haha. I guess all the good bits ...,0,3
4,family guy funny,1,4


In [11]:
print(partitions.shape)
print(sample_df["label"].value_counts())
print(sample_df["fold"].value_counts())

(200000, 2)
label
1    100019
0     99981
Name: count, dtype: int64
fold
1    66667
0    66667
2    66666
Name: count, dtype: int64


## Conectarse al mlflow

In [12]:
!pip install mlflow -q

In [21]:
import mlflow

mlflow.set_tracking_uri("http://ec2-100-26-91-142.compute-1.amazonaws.com:5000")
mlflow.set_experiment("nlp-lab2-sentiment140")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1789655162210, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789655162210, lifecycle_stage='active', name='nlp-lab2-sentiment140', tags={}, trace_location=None, workspace='default'>

## Run de registro

In [18]:
with mlflow.start_run(run_name="protocol") as run:
    mlflow.set_tag("lab_run_type", "protocol")

    mlflow.log_param("dataset_id", "adilbekovich/Sentiment140Twitter")
    mlflow.log_param("dataset_revision", "b6037e127257d95b9b23d31f78b264b9ebe697fd")
    mlflow.log_param("sampling_strategy", "stratified")
    mlflow.log_param("sample_size", 200000)
    mlflow.log_param("random_seed", 42)
    mlflow.log_param("cv_strategy", "StratifiedKFold")
    mlflow.log_param("cv_folds", 3)
    mlflow.log_param("cv_shuffle", True)

    mlflow.log_artifact("protocol/partitions.csv", artifact_path="protocol")
    mlflow.log_artifact("protocol/members.csv", artifact_path="protocol")

    protocol_run_id = run.info.run_id
    print("Protocol run ID:", protocol_run_id)

Protocol run ID: 5159d4c11e934382ad32d19269f79112
🏃 View run protocol at: http://ec2-54-160-133-217.compute-1.amazonaws.com:5000/#/experiments/1/runs/5159d4c11e934382ad32d19269f79112
🧪 View experiment at: http://ec2-54-160-133-217.compute-1.amazonaws.com:5000/#/experiments/1


Registra el dataset, muestra folds e integrantes, todavia no se entrena nada

## Funcion generica de evaluacion

In [2]:
!pip install scikit-learn mlflow datasets -q

In [29]:
def evaluate_pipeline(build_pipeline_fn, train_df, partitions_df):
    # Solo nos quedamos con las columnas necesarias de train_df, sin la columna 'fold' propia
    base = train_df[["index", "text", "label"]]
    merged = base.merge(partitions_df, on="index", how="inner")
    assert len(merged) == len(partitions_df), "La muestra no coincide con partitions.csv"

    fold_scores = []
    for fold_k in sorted(merged["fold"].unique()):
        train_part = merged[merged["fold"] != fold_k]
        val_part = merged[merged["fold"] == fold_k]

        X_train, y_train = train_part["text"].tolist(), train_part["label"].values
        X_val, y_val = val_part["text"].tolist(), val_part["label"].values

        model = build_pipeline_fn(X_train, y_train)
        y_pred = model.predict(X_val)

        f1 = f1_score(y_val, y_pred, average="macro")
        fold_scores.append(f1)
        print(f"  Fold {fold_k}: Macro-F1 = {f1:.4f}")

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores, ddof=0)
    return fold_scores, mean_f1, std_f1

## TO - clase mas frecuente

In [30]:
class MostFrequentClassifier:
    def __init__(self):
        self.majority_class = None

    def fit(self, X, y):
        counts = pd.Series(y).value_counts()
        if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
            self.majority_class = 0  # empate -> negative
        else:
            self.majority_class = counts.idxmax()
        return self

    def predict(self, X):
        return np.full(len(X), self.majority_class)


def build_t0(X_train, y_train):
    model = MostFrequentClassifier()
    model.fit(X_train, y_train)
    return model

print("Evaluando T0...")
t0_scores, t0_mean, t0_std = evaluate_pipeline(build_t0, sample_df, partitions)
print(f"T0 -> mean={t0_mean:.4f}, std={t0_std:.4f}")

Evaluando T0...
  Fold 0: Macro-F1 = 0.3334
  Fold 1: Macro-F1 = 0.3334
  Fold 2: Macro-F1 = 0.3334
T0 -> mean=0.3334, std=0.0000


## Registrar TO en MLflow

In [32]:
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("nlp-lab2-sentiment140")
runs = client.search_runs(exp.experiment_id, filter_string="tags.lab_run_type = 'protocol'")
protocol_run_id = runs[0].info.run_id
print("Protocol run ID recuperado:", protocol_run_id)

Protocol run ID recuperado: 5159d4c11e934382ad32d19269f79112


In [33]:
import json

t0_config = {
    "preprocessing": None,
    "representation": None,
    "classifier": {
        "type": "most_frequent",
        "library": "custom",
        "library_version": "1.0",
        "parameters": {}
    }
}

with mlflow.start_run(run_name="T0") as run:
    mlflow.set_tag("lab_run_type", "experiment")
    mlflow.set_tag("lab_protocol_run_id", protocol_run_id)
    mlflow.set_tag("lab_experiment_id", "T0")
    mlflow.set_tag("lab_stage", "reference")
    mlflow.set_tag("lab_member_id", "E01")  # ajusta a tu member_id real
    mlflow.set_tag("lab_configuration_id", "CFG_T0")
    mlflow.set_tag("notebook_arn", meta.get("ResourceArn"))

    for i, f1 in enumerate(t0_scores):
        mlflow.log_metric(f"macro_f1_fold_{i}", f1)
    mlflow.log_metric("macro_f1_mean", t0_mean)
    mlflow.log_metric("macro_f1_std", t0_std)

    with open("configuration.json", "w") as f:
        json.dump(t0_config, f, indent=2)
    mlflow.log_artifact("configuration.json", artifact_path="run")

    mlflow.log_artifact("provenance/sagemaker-resource-metadata.json", artifact_path="provenance")

    t0_run_id = run.info.run_id
    print("T0 run ID:", t0_run_id)

T0 run ID: fb9bee918f744b2bb15645d411fdf6b0
🏃 View run T0 at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1/runs/fb9bee918f744b2bb15645d411fdf6b0
🧪 View experiment at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1


## BO-BoW + regresion logistica

In [34]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import re

def preprocess_b0(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "url", text)
    text = re.sub(r"@\w+", "user", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_b0(X_train, y_train):
    X_train_proc = [preprocess_b0(t) for t in X_train]
    pipeline = Pipeline([
        ("vectorizer", CountVectorizer(ngram_range=(1, 1))),
        ("clf", LogisticRegression())
    ])
    pipeline.fit(X_train_proc, y_train)

    class Wrapped:
        def predict(self, X):
            X_proc = [preprocess_b0(t) for t in X]
            return pipeline.predict(X_proc)
    return Wrapped()

print("Evaluando B0...")
b0_scores, b0_mean, b0_std = evaluate_pipeline(build_b0, sample_df, partitions)
print(f"B0 -> mean={b0_mean:.4f}, std={b0_std:.4f}")

Evaluando B0...


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 0: Macro-F1 = 0.7817


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 1: Macro-F1 = 0.7836


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 2: Macro-F1 = 0.7819
B0 -> mean=0.7824, std=0.0009


## Registrar BO en MLflow

In [36]:
import sklearn

b0_config = {
    "preprocessing": {
        "lowercase": True,
        "url": "token:url",
        "mention": "token:user",
        "whitespace": "normalize",
        "stopwords": "keep",
        "negators": [],
        "lemmatize": False,
        "elongation": "keep",
        "elongation_spec": None,
        "emoji": "keep",
        "emoji_spec": None,
        "resources": {},
        "additional": {}
    },
    "representation": {
        "type": "bow",
        "ngram_range": [1, 1],
        "library": "sklearn",
        "library_version": sklearn.__version__,
        "spacy_model": None,
        "spacy_model_version": None,
        "parameters": {}
    },
    "classifier": {
        "type": "logistic_regression",
        "library": "sklearn",
        "library_version": sklearn.__version__,
        "parameters": {}
    }
}

with mlflow.start_run(run_name="B0") as run:
    mlflow.set_tag("lab_run_type", "experiment")
    mlflow.set_tag("lab_protocol_run_id", protocol_run_id)
    mlflow.set_tag("lab_experiment_id", "B0")
    mlflow.set_tag("lab_stage", "baseline")
    mlflow.set_tag("lab_member_id", "E01")
    mlflow.set_tag("lab_configuration_id", "CFG_B0")
    mlflow.set_tag("notebook_arn", meta.get("ResourceArn"))

    for i, f1 in enumerate(b0_scores):
        mlflow.log_metric(f"macro_f1_fold_{i}", f1)
    mlflow.log_metric("macro_f1_mean", b0_mean)
    mlflow.log_metric("macro_f1_std", b0_std)

    with open("configuration.json", "w") as f:
        json.dump(b0_config, f, indent=2)
    mlflow.log_artifact("configuration.json", artifact_path="run")

    mlflow.log_artifact("provenance/sagemaker-resource-metadata.json", artifact_path="provenance")

    b0_run_id = run.info.run_id
    print("B0 run ID:", b0_run_id)

B0 run ID: 11b95d08b9af461aaf0e29fba3c6fdba
🏃 View run B0 at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1/runs/11b95d08b9af461aaf0e29fba3c6fdba
🧪 View experiment at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1


## Instalar spacy

In [37]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## Carga de spacy y definir los navegadores del equipo

In [38]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Negadores que el equipo decide preservar aunque se eliminen stopwords
NEGATORS = ["not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nor"]

## Funcion de preprocesamiento 

In [39]:
import re

def preprocess_text(text, stopwords_mode="keep", lemmatize=False,
                     elongation="keep", emoji_mode="keep"):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "url", text)
    text = re.sub(r"@\w+", "user", text)

    if elongation == "normalize":
        # reduce repeticiones de 3+ caracteres a 2 (ej: "soooo" -> "soo")
        text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    if emoji_mode == "text":
        import emoji as emoji_lib
        text = emoji_lib.demojize(text, delimiters=(" ", " "))

    text = re.sub(r"\s+", " ", text).strip()

    if stopwords_mode == "keep" and not lemmatize:
        return text

    doc = nlp(text)
    tokens = []
    for token in doc:
        word = token.text
        is_stop = token.is_stop or word in NEGATORS  # spaCy no siempre marca "n't" como stop

        if stopwords_mode == "remove" and is_stop:
            continue
        if stopwords_mode == "remove_preserve_negation" and is_stop and word not in NEGATORS:
            continue

        if lemmatize:
            word = token.lemma_

        tokens.append(word)

    return " ".join(tokens)

In [40]:
!pip install emoji -q

## Funcion para construir y evaluar pipeline BoW+LogReg con procesamiento

In [41]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def make_build_fn(preprocess_kwargs):
    def build_fn(X_train, y_train):
        X_train_proc = [preprocess_text(t, **preprocess_kwargs) for t in X_train]
        pipeline = Pipeline([
            ("vectorizer", CountVectorizer(ngram_range=(1, 1))),
            ("clf", LogisticRegression())
        ])
        pipeline.fit(X_train_proc, y_train)

        class Wrapped:
            def predict(self, X):
                X_proc = [preprocess_text(t, **preprocess_kwargs) for t in X]
                return pipeline.predict(X_proc)
        return Wrapped()
    return build_fn

## Funcion generica de registro en Mlflow

In [42]:
def log_experiment_run(run_name, lab_experiment_id, lab_stage, configuration_id,
                        config_json, scores, mean_f1, std_f1, member_id="E01"):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("lab_run_type", "experiment")
        mlflow.set_tag("lab_protocol_run_id", protocol_run_id)
        mlflow.set_tag("lab_experiment_id", lab_experiment_id)
        mlflow.set_tag("lab_stage", lab_stage)
        mlflow.set_tag("lab_member_id", member_id)
        mlflow.set_tag("lab_configuration_id", configuration_id)
        mlflow.set_tag("notebook_arn", meta.get("ResourceArn"))

        for i, f1 in enumerate(scores):
            mlflow.log_metric(f"macro_f1_fold_{i}", f1)
        mlflow.log_metric("macro_f1_mean", mean_f1)
        mlflow.log_metric("macro_f1_std", std_f1)

        with open("configuration.json", "w") as f:
            json.dump(config_json, f, indent=2)
        mlflow.log_artifact("configuration.json", artifact_path="run")

        mlflow.log_artifact("provenance/sagemaker-resource-metadata.json", artifact_path="provenance")

        print(f"{lab_experiment_id} run ID:", run.info.run_id)
        return run.info.run_id

## Base de configuracion

In [43]:
import copy

def base_config():
    return {
        "preprocessing": {
            "lowercase": True, "url": "token:url", "mention": "token:user",
            "whitespace": "normalize", "stopwords": "keep", "negators": [],
            "lemmatize": False, "elongation": "keep", "elongation_spec": None,
            "emoji": "keep", "emoji_spec": None, "resources": {}, "additional": {}
        },
        "representation": {
            "type": "bow", "ngram_range": [1, 1], "library": "sklearn",
            "library_version": sklearn.__version__, "spacy_model": None,
            "spacy_model_version": None, "parameters": {}
        },
        "classifier": {
            "type": "logistic_regression", "library": "sklearn",
            "library_version": sklearn.__version__, "parameters": {}
        }
    }

## P_STOPWORDS

In [44]:
config = base_config()
config["preprocessing"]["stopwords"] = "remove"

build_fn = make_build_fn({"stopwords_mode": "remove"})
print("Evaluando P_STOPWORDS...")
scores, mean_f1, std_f1 = evaluate_pipeline(build_fn, sample_df, partitions)
print(f"P_STOPWORDS -> mean={mean_f1:.4f}, std={std_f1:.4f}")

log_experiment_run("P_STOPWORDS", "P_STOPWORDS", "preprocessing", "CFG_P_STOPWORDS",
                    config, scores, mean_f1, std_f1)

Evaluando P_STOPWORDS...


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 0: Macro-F1 = 0.7579


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 1: Macro-F1 = 0.7608


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 2: Macro-F1 = 0.7571
P_STOPWORDS -> mean=0.7586, std=0.0016
P_STOPWORDS run ID: 8f4cc24b5ae74bd5a654174b6e1620b2
🏃 View run P_STOPWORDS at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1/runs/8f4cc24b5ae74bd5a654174b6e1620b2
🧪 View experiment at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1


'8f4cc24b5ae74bd5a654174b6e1620b2'